In [1]:
# =============================================================================
# TASK 5 — Dominant SMOD L1 class per ADM2 by "geographic area" proxy (cell counts)
# Uses SMOD L1 raster in native SMOD CRS (Mollweide / ESRI:54009), no harmonisation to WorldPop.
#
# Outputs (in outputs/):
#  - 05_smodL1_cellcounts_by_adm2.csv
#  - 05_smodL1_dominant_summary.csv
#  - 05_smodL1_dominant_report.json
#  - 05_smodL1_dominant_map.png
#  - 05_smodL1_cellcount_stackedbar.png
#  - 05_adm2_with_dominant.gpkg (optional, but useful)
# =============================================================================

from pathlib import Path
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats

import matplotlib.pyplot as plt


# -----------------------------------------------------------------------------
# USER SETTINGS — project structure
# -----------------------------------------------------------------------------
ROOT = Path(r"C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton")
DATA = ROOT / "data_raw"
OUT  = ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

# Inputs
boundary_path = DATA / "ZMB_adm2_gadm41_Copperbelt.shp"
smod_l1_path  = OUT / "04_smod_L1_from_L2_54009_1km_INT16.tif"   # from Task 4 (recode only)

# Output files
out_csv   = OUT / "05_smodL1_cellcounts_by_adm2.csv"
out_sum   = OUT / "05_smodL1_dominant_summary.csv"
out_json  = OUT / "05_smodL1_dominant_report.json"
out_map   = OUT / "05_smodL1_dominant_map.png"
out_bar   = OUT / "05_smodL1_cellcount_stackedbar.png"
out_gpkg  = OUT / "05_adm2_with_dominant.gpkg"

# Options
ALL_TOUCHED = False  # conservative cell counting; set True if you want border-touching cells included
NAME_FIELD  = "NAME_2"  # district name
NODATA_L1   = -200      # nodata in your L1 raster

run_time_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")


# -----------------------------------------------------------------------------
# 1) Read inputs + reproject boundaries to SMOD CRS
# -----------------------------------------------------------------------------
gdf = gpd.read_file(boundary_path)

if gdf.crs is None:
    raise ValueError("Boundary CRS missing. Define CRS before running zonal/cell-count stats.")

if NAME_FIELD not in gdf.columns:
    raise ValueError(f"{NAME_FIELD} not found in boundary attributes. Available: {list(gdf.columns)}")

with rasterio.open(smod_l1_path) as src:
    smod_crs = src.crs
    smod_transform = src.transform
    smod_shape = (src.height, src.width)
    smod_dtype = src.dtypes[0]
    smod_nodata = src.nodata

# Ensure nodata is what you expect
if smod_nodata is None:
    smod_nodata = NODATA_L1

# Reproject polygons to raster CRS (important for correct rasterisation)
gdf_smod = gdf.to_crs(smod_crs)

print("Inputs:")
print(" - ADM2:", boundary_path.name, "| features:", len(gdf_smod), "| CRS:", gdf_smod.crs)
print(" - SMOD L1:", smod_l1_path.name, "| CRS:", smod_crs, "| dtype:", smod_dtype, "| nodata:", smod_nodata, "| shape:", smod_shape)
print(" - all_touched:", ALL_TOUCHED)
print()


# -----------------------------------------------------------------------------
# 2) Zonal "histogram" of categorical values (cell counts per class)
# -----------------------------------------------------------------------------
# zonal_stats with categorical=True returns dict {value: count} for each polygon
zs = zonal_stats(
    vectors=gdf_smod,
    raster=str(smod_l1_path),
    categorical=True,
    nodata=smod_nodata,
    all_touched=ALL_TOUCHED
)

# Helper: get count for a class code from dict
def get_count(d, code):
    return int(d.get(code, 0)) if isinstance(d, dict) else 0

rows = []
for i, d in enumerate(zs):
    nm = str(gdf_smod.iloc[i][NAME_FIELD])

    c_rural = get_count(d, 1)
    c_urbcl = get_count(d, 2)
    c_urbct = get_count(d, 3)
    total = c_rural + c_urbcl + c_urbct

    # proportions (avoid /0)
    pr = (c_rural / total) if total > 0 else np.nan
    pu = (c_urbcl / total) if total > 0 else np.nan
    pc = (c_urbct / total) if total > 0 else np.nan

    # dominant class with tie handling
    counts = {1: c_rural, 2: c_urbcl, 3: c_urbct}
    maxv = max(counts.values()) if total > 0 else 0
    winners = [k for k, v in counts.items() if v == maxv and total > 0]

    if total == 0:
        dom_code = np.nan
        dom_label = "NO_DATA"
        tie = False
    elif len(winners) == 1:
        dom_code = winners[0]
        dom_label = {1: "Rural", 2: "Urban cluster", 3: "Urban centre"}[dom_code]
        tie = False
    else:
        dom_code = np.nan
        dom_label = "TIE"
        tie = True

    rows.append({
        "district": nm,
        "cells_rural": c_rural,
        "cells_urban_cluster": c_urbcl,
        "cells_urban_centre": c_urbct,
        "cells_total": total,
        "prop_rural": pr,
        "prop_urban_cluster": pu,
        "prop_urban_centre": pc,
        "dominant_class": dom_label,
        "dominant_code": dom_code,
        "tie_for_dominant": tie
    })

df = pd.DataFrame(rows)

# Sort for readability: largest total cells first
df = df.sort_values(["cells_total", "district"], ascending=[False, True]).reset_index(drop=True)

df.to_csv(out_csv, index=False)
print("Saved:", out_csv)


# -----------------------------------------------------------------------------
# 3) Summary counts: how many districts dominated by each class?
# -----------------------------------------------------------------------------
summary = (df["dominant_class"]
           .value_counts(dropna=False)
           .rename_axis("dominant_class")
           .reset_index(name="n_adm2"))

summary["share"] = summary["n_adm2"] / len(df)
summary.to_csv(out_sum, index=False)
print("Saved:", out_sum)

# Extract counts asked by the question
n_urban_centre_dom = int((df["dominant_class"] == "Urban centre").sum())
n_rural_dom        = int((df["dominant_class"] == "Rural").sum())

print("\nTask 5 answer (boundary as provided):")
print(f" - ADM2 dominated by Urban centre (by SMOD L1 cell count): {n_urban_centre_dom}")
print(f" - ADM2 dominated by Rural (by SMOD L1 cell count): {n_rural_dom}")
print()


# -----------------------------------------------------------------------------
# 4) Visualisation A: Map of dominant class per ADM2
# -----------------------------------------------------------------------------
# Join dominant class back to polygons
gdf_out = gdf_smod.copy()
gdf_out = gdf_out.merge(df[["district", "dominant_class"]], left_on=NAME_FIELD, right_on="district", how="left")

# Save optional GeoPackage for inspection in QGIS
try:
    gdf_out.to_file(out_gpkg, layer="adm2_dominant_smod_l1", driver="GPKG")
    print("Saved:", out_gpkg)
except Exception as e:
    print("GeoPackage write skipped (non-fatal):", e)

# Plot dominant class map (categorical)
fig, ax = plt.subplots(figsize=(8, 7))
gdf_out.plot(column="dominant_class", ax=ax, legend=True, linewidth=0.8, edgecolor="black")
ax.set_title("Dominant SMOD L1 class by ADM2 (dominance by cell count)")
ax.set_axis_off()
plt.tight_layout()
plt.savefig(out_map, dpi=300)
plt.close(fig)

print("Saved:", out_map)


# -----------------------------------------------------------------------------
# 5) Visualisation B: Stacked bar (cell counts by class per ADM2)
# -----------------------------------------------------------------------------
# Make a tidy plotting frame
plot_df = df.set_index("district")[["cells_rural", "cells_urban_cluster", "cells_urban_centre"]]

fig, ax = plt.subplots(figsize=(10, 5))
plot_df.plot(kind="bar", stacked=True, ax=ax)  # uses default colours
ax.set_ylabel("Number of SMOD L1 grid cells")
ax.set_title("ADM2 settlement composition by SMOD L1 (cell counts)")
ax.tick_params(axis="x", labelrotation=60)
plt.tight_layout()
plt.savefig(out_bar, dpi=300)
plt.close(fig)

print("Saved:", out_bar)


# -----------------------------------------------------------------------------
# 6) Reproducibility JSON artefact
# -----------------------------------------------------------------------------
report = {
    "run_time_utc": run_time_utc,
    "inputs": {
        "boundary_path": str(boundary_path),
        "smod_l1_path": str(smod_l1_path),
        "boundary_crs_original": str(gdf.crs),
        "boundary_crs_used": str(gdf_smod.crs),
        "smod_crs": str(smod_crs),
        "smod_dtype": str(smod_dtype),
        "smod_nodata": smod_nodata,
        "all_touched": ALL_TOUCHED
    },
    "outputs": {
        "per_adm2_csv": str(out_csv),
        "dominant_summary_csv": str(out_sum),
        "dominant_map_png": str(out_map),
        "stacked_bar_png": str(out_bar),
        "adm2_gpkg_optional": str(out_gpkg)
    },
    "results": {
        "n_adm2": int(len(df)),
        "dominant_counts": summary.to_dict(orient="records"),
        "answer_counts": {
            "urban_centre_dominant": n_urban_centre_dom,
            "rural_dominant": n_rural_dom
        }
    },
    "notes": [
        "Dominance is based on SMOD L1 cell counts inside each ADM2 polygon (proxy for area).",
        "NoData (-200) and Water (already NoData in L1) are excluded from counts.",
        "Boundaries are used as provided (no corrections applied).",
        "If tie_for_dominant=True, the ADM2 has equal maximum counts across ≥2 classes and is labelled TIE."
    ]
}

with open(out_json, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("Saved:", out_json)


# -----------------------------------------------------------------------------
# 7) Copy-friendly interview snippet
# -----------------------------------------------------------------------------
print("\n--- Interview notes (Task 5) ---")
print("Reprojected ADM2 polygons to SMOD Mollweide CRS and computed per-district SMOD L1 cell counts (categorical zonal histogram).")
print(f"Urban centre dominant ADM2: {n_urban_centre_dom}; Rural dominant ADM2: {n_rural_dom}.")
print("Saved a per-ADM2 table (cell counts + proportions + dominant class), a dominance summary table, and two visuals (dominant-class map + stacked bars).")
print("--------------------------------\n")


Inputs:
 - ADM2: ZMB_adm2_gadm41_Copperbelt.shp | features: 12 | CRS: PROJCS["World_Mollweide",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Mollweide"],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
 - SMOD L1: 04_smod_L1_from_L2_54009_1km_INT16.tif | CRS: ESRI:54009 | dtype: int16 | nodata: -200.0 | shape: (1000, 1000)
 - all_touched: False

Saved: C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\05_smodL1_cellcounts_by_adm2.csv
Saved: C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\05_smodL1_dominant_summary.csv

Task 5 answer (boundary as provided):
 - ADM2 domin